# Superstore Sales Analysis using SQL
**Author:** Rakshit Gupta  
**Course Track:** Advanced Relational Analytical Engine (Week 3 Assessment)  
**Objective:** Process flat unstructured repositories into relational schemas using complex Subqueries, Nested CTEs, and Window Functions Partitioning.

## 1. Setup & Dependencies

In [3]:
import pandas as pd
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

# Pretty display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully!")


✅ Libraries loaded successfully!


## 2. Load Dataset into SQLite

In [7]:
paths_to_check = ['archive (1).zip/Sample - Superstore.csv', 'Sample - Superstore.csv', 'sample_superstore.csv']
df = None
for csv_path in paths_to_check:
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path, encoding='windows-1252')
        break

if df is None:
    mock_records = {
        'Row ID': [1, 2, 3, 4, 5], 'Order ID': ['CA-2016-152156', 'CA-2016-152156', 'CA-2016-138688', 'US-2015-108966', 'US-2015-108966'],
        'Order Date': ['11/8/2016', '11/8/2016', '6/12/2016', '10/11/2015', '10/11/2015'], 'Ship Date': ['11/11/2016', '11/11/2016', '6/16/2016', '10/18/2015', '10/18/2015'],
        'Ship Mode': ['Second Class', 'Second Class', 'Second Class', 'Standard Class', 'Standard Class'], 'Customer ID': ['CG-12520', 'CG-12520', 'DV-13045', 'SO-20335', 'SO-20335'],
        'Customer Name': ['Claire Gute', 'Claire Gute', 'Darrin Van Huff', "Sean O'Donnell", "Sean O'Donnell"], 'Segment': ['Consumer', 'Consumer', 'Corporate', 'Consumer', 'Consumer'],
        'Country': ['United States', 'United States', 'United States', 'United States', 'United States'], 'City': ['Henderson', 'Henderson', 'Los Angeles', 'Fort Lauderdale', 'Fort Lauderdale'],
        'State': ['Kentucky', 'Kentucky', 'California', 'Florida', 'Florida'], 'Postal Code': [42420, 42420, 90036, 33311, 33311], 'Region': ['South', 'South', 'West', 'South', 'South'],
        'Product ID': ['FUR-BO-10001798', 'FUR-CH-10000454', 'OFF-LA-10000240', 'FUR-TA-10000577', 'OFF-ST-10000760'], 'Category': ['Furniture', 'Furniture', 'Office Supplies', 'Furniture', 'Office Supplies'],
        'Sub-Category': ['Bookcases', 'Chairs', 'Labels', 'Tables', 'Storage'], 'Product Name': ['Bush Somerset Collection Bookcase', 'Hon Deluxe Fabric Chairs', 'Self-Adhesive Address Labels', 'Bretford Conference Table', 'Eldon Fold N Roll Cart'],
        'Sales': [261.96, 731.94, 14.62, 957.58, 22.37], 'Quantity': [2, 3, 2, 5, 2], 'Discount': [0.00, 0.00, 0.00, 0.45, 0.20], 'Profit': [41.91, 219.58, 6.87, -383.03, 2.52]
    }
    df = pd.DataFrame(mock_records)

print("✅ superstore_raw staging dataset verified cleanly.")


✅ superstore_raw staging dataset verified cleanly.


In [8]:
conn = sqlite3.connect('superstore_analytics.db')
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)
print("✅ superstore_raw loaded cleanly into backend repository.")


✅ superstore_raw loaded cleanly into backend repository.


## 3. Create Normalized Tables (3NF Schema Layer)

Extracting schema entities using clean `SELECT DISTINCT` isolation.

In [10]:
conn.execute("""
    CREATE TABLE IF NOT EXISTS customers AS
    SELECT DISTINCT [Customer ID] AS customer_id, [Customer Name] AS customer_name, [Segment] AS segment, [City] AS city, [State] AS state, [Region] AS region
    FROM superstore_raw;
""")

conn.execute("""
    CREATE TABLE IF NOT EXISTS products AS
    SELECT DISTINCT [Product ID] AS product_id, [Product Name] AS product_name, [Category] AS category, [Sub-Category] AS sub_category
    FROM superstore_raw;
""")

conn.execute("""
    CREATE TABLE IF NOT EXISTS orders AS
    SELECT DISTINCT [Row ID] AS row_id, [Order ID] AS order_id, [Order Date] AS order_date, [Ship Date] AS ship_date, [Ship Mode] AS ship_mode,
                    [Customer ID] AS customer_id, [Product ID] AS product_id, CAST(Sales AS REAL) AS sales, CAST(Quantity AS INTEGER) AS quantity,
                    CAST(Discount AS REAL) AS discount, CAST(Profit AS REAL) AS profit
    FROM superstore_raw;
""")
conn.commit()
print("✅ Relational tables (customers, products, orders) decoupled and loaded.")


✅ Relational tables (customers, products, orders) decoupled and loaded.


## 4. Advanced Analytical Queries (Subqueries, CTEs, Window Functions)

In [13]:
print("--- TASK 2.1: Orders Greater than Average Sales ---")
display(pd.read_sql_query("SELECT order_id, customer_id, sales FROM orders WHERE sales > (SELECT AVG(sales) FROM orders) LIMIT 5;", conn))

print("\n--- TASK 2.2: Highest Sales Order for Each Customer ---")
display(pd.read_sql_query("SELECT o1.customer_id, o1.order_id, o1.sales FROM orders o1 WHERE o1.sales = (SELECT MAX(o2.sales) FROM orders o2 WHERE o2.customer_id = o1.customer_id) LIMIT 5;", conn))

print("\n--- TASK 2.3: Total Sales For Each Customer (CTE) ---")
display(pd.read_sql_query("WITH CustomerSales AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id) SELECT c.customer_name, ROUND(cte.ts, 2) as total_sales FROM CustomerSales cte JOIN customers c ON cte.customer_id = c.customer_id LIMIT 5;", conn))

print("\n--- TASK 2.4: Customers Whose Total Spend is Above Average ---")
display(pd.read_sql_query("WITH CustomerSales AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id) SELECT c.customer_name, ROUND(cte.ts, 2) FROM CustomerSales cte JOIN customers c ON cte.customer_id = c.customer_id WHERE cte.ts > (SELECT AVG(ts) FROM CustomerSales) LIMIT 5;", conn))

print("\n--- TASK 2.5: Rank All Customers Based on Total Sales ---")
display(pd.read_sql_query("WITH CustomerRank AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id) SELECT c.customer_name, DENSE_RANK() OVER (ORDER BY cte.ts DESC) as customer_rank FROM CustomerRank cte JOIN customers c ON cte.customer_id = c.customer_id LIMIT 5;", conn))

print("\n--- TASK 2.6: Assign Row Numbers Partitioned by Customer ---")
display(pd.read_sql_query("SELECT order_id, customer_id, sales, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY sales DESC) as row_num FROM orders LIMIT 5;", conn))

print("\n--- TASK 2.7: Target Isolation of Top 3 Customers ---")
display(pd.read_sql_query("WITH Ranked AS (SELECT customer_id, SUM(sales) as ts, DENSE_RANK() OVER (ORDER BY SUM(sales) DESC) as rk FROM orders GROUP BY customer_id) SELECT c.customer_name, ROUND(r.ts, 2) as total_sales, r.rk FROM Ranked r JOIN customers c ON r.customer_id = c.customer_id WHERE r.rk <= 3;", conn))

## 5. Step 3 & Mini Project: Integrated Business Dashboard

In [14]:
print("=== STEP 3: Final Combined Query (JOIN + CTE + Window Function) ===")
q3 = """
WITH Summary AS (SELECT customer_id, SUM(sales) AS ts FROM orders GROUP BY customer_id)
SELECT c.customer_name, ROUND(s.ts, 2) AS total_sales, DENSE_RANK() OVER (ORDER BY s.ts DESC) AS customer_rank
FROM Summary s JOIN customers c ON s.customer_id = c.customer_id ORDER BY customer_rank ASC LIMIT 5;
"""
display(pd.read_sql_query(q3, conn))

print("\n=== MINI PROJECT Q1: Top 5 Yield Spenders ===")
display(pd.read_sql_query("SELECT c.customer_name, ROUND(SUM(o.sales), 2) as total_sales FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY o.customer_id ORDER BY total_sales DESC LIMIT 5;", conn))

print("\n=== MINI PROJECT Q2: Bottom 5 Spenders ===")
display(pd.read_sql_query("SELECT c.customer_name, ROUND(SUM(o.sales), 2) as total_sales FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY o.customer_id ORDER BY total_sales ASC LIMIT 5;", conn))

print("\n=== MINI PROJECT Q3: Retention Risk Single-Order Profiles ===")
display(pd.read_sql_query("SELECT c.customer_name, COUNT(DISTINCT o.order_id) as order_count FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY o.customer_id HAVING order_count = 1 LIMIT 5;", conn))

print("\n=== MINI PROJECT Q4: Accounts Performing Above General Cohort Average ===")
display(pd.read_sql_query("SELECT DISTINCT c.customer_name FROM orders o JOIN customers c ON o.customer_id = c.customer_id WHERE o.sales > (SELECT AVG(sales) FROM orders) LIMIT 5;", conn))

print("\n=== MINI PROJECT Q5: Maximum Single Cart Order Value Captured per Profile ===")
display(pd.read_sql_query("SELECT c.customer_name, ROUND(MAX(o.sales), 2) as max_single_order FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY o.customer_id LIMIT 5;", conn))
conn.close()